# C — CVAE couleur
**Amjad AAYD et [information masquée] — [information masquée]**

Adaptation du TP AE / VAE / CVAE / PixelCNN à **CIFAR-10**, selon la demande orale
rapportée par l'étudiant. « CIFAR dix » est l'interprétation retenue du nom entendu.
Le PDF d'origine utilise FashionMNIST et MNIST ; cette version utilise CIFAR-10
pour les quatre blocs. Les images restent en couleur, sans conversion en gris.

Lire de haut en bas : **objectif → code commenté → sortie → explication**.
Les classes Python ne sont utilisées que pour définir les réseaux PyTorch.
Les boucles d'apprentissage restent visibles et explicites.

In [ ]:
# Importer uniquement les bibliothèques utilisées dans ce notebook.
from pathlib import Path
import json
import time
import random
import numpy as np
import torch
from torch import nn
from torch.nn import functional as F
from torch.utils.data import DataLoader, TensorDataset
from torchvision.datasets import CIFAR10
import matplotlib
matplotlib.use("Agg")
import matplotlib.pyplot as plt
from IPython.display import display, Image

# Fixer les tirages pour pouvoir refaire une expérience comparable.
random.seed(42)
np.random.seed(42)
torch.manual_seed(42)
torch.set_num_threads(6)

# Utiliser le GPU si ce PC en possède un ; sinon utiliser le processeur.
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print("Calcul sur :", device)
if device.type == "cuda":
    print(torch.cuda.get_device_name(0))

# Tous les fichiers créés restent dans le dossier de ce TP.
racine = Path.cwd()
(racine / "figures").mkdir(exist_ok=True)
(racine / "models").mkdir(exist_ok=True)
debut = time.time()

## Charger les images couleur
Une image contient **3 × 32 × 32 = 3 072 nombres** : rouge, vert et bleu.
La division par 255 place les intensités dans [0,1]. Le test est chargé séparément
et ne sert jamais à ajuster les poids. Les classes ne supervisent que le CVAE.

In [ ]:
# Télécharger les vraies images CIFAR-10 avec torchvision.
# Au prochain lancement, les fichiers locaux sont réutilisés.
train = CIFAR10(root="data", train=True, download=True)
test = CIFAR10(root="data", train=False, download=True)

# CIFAR stocke [image, hauteur, largeur, couleur].
# PyTorch attend [image, couleur, hauteur, largeur].
x_train = torch.tensor(train.data).permute(0, 3, 1, 2).float() / 255.0
x_test = torch.tensor(test.data).permute(0, 3, 1, 2).float() / 255.0
y_train = torch.tensor(train.targets)
y_test = torch.tensor(test.targets)

# Former des lots de 128 images pour l'apprentissage.
donnees_train = TensorDataset(x_train, y_train)
chargeur = DataLoader(donnees_train, batch_size=128, shuffle=True)
classes = ["avion", "voiture", "oiseau", "chat", "cerf",
           "chien", "grenouille", "cheval", "bateau", "camion"]
print("Apprentissage :", x_train.shape)
print("Test :", x_test.shape)
print("Classes :", classes)
assert x_train.shape == (50000, 3, 32, 32)
assert x_test.shape == (10000, 3, 32, 32)

In [ ]:
# Cette fonction sert seulement à afficher et sauvegarder une grille de couleurs.
# permute remet les couleurs en dernier, comme attendu par matplotlib.
def afficher(images, titre, fichier, colonnes=8):
    images = images.detach().cpu().clamp(0, 1)
    lignes = (len(images) + colonnes - 1) // colonnes
    figure, axes = plt.subplots(lignes, colonnes,
                               figsize=(colonnes * 1.4, lignes * 1.5),
                               squeeze=False)
    for numero, axe in enumerate(axes.flat):
        axe.axis("off")
        if numero < len(images):
            axe.imshow(images[numero].permute(1, 2, 0))
    figure.suptitle(titre)
    figure.tight_layout()
    chemin = racine / "figures" / fichier
    figure.savefig(chemin, dpi=140, bbox_inches="tight")
    plt.close(figure)
    display(Image(filename=str(chemin)))

# Une seconde petite fonction affiche une courbe déjà calculée.
def courbe(valeurs, titre, fichier):
    plt.figure(figsize=(6, 3))
    plt.plot(range(1, len(valeurs) + 1), valeurs, marker="o")
    plt.xlabel("Époque")
    plt.ylabel("Perte moyenne")
    plt.title(titre)
    plt.grid(alpha=0.2)
    plt.tight_layout()
    chemin = racine / "figures" / fichier
    plt.savefig(chemin, dpi=140)
    plt.close()
    display(Image(filename=str(chemin)))

## Conditionner par une catégorie d'objet
Dans le TP original, y désignait un chiffre MNIST. Ici, y désigne l'une des dix
catégories CIFAR-10. Le code latent a 8 valeurs ; le one-hot en a 10.
Le code garde toutes les concaténations visibles pour pouvoir les expliquer.

In [ ]:
# Un CVAE simple : ajouter y à l'encodeur et au décodeur, sans classe VAE générale.
class CVAE(nn.Module):
    def __init__(self):
        super().__init__()
        self.encodeur = nn.Sequential(
            nn.Conv2d(3, 32, 4, 2, 1), nn.ReLU(),
            nn.Conv2d(32, 64, 4, 2, 1), nn.ReLU(), nn.Flatten()
        )
        self.cachee = nn.Linear(64 * 8 * 8 + 10, 256)
        self.moyenne = nn.Linear(256, 8)
        self.log_variance = nn.Linear(256, 8)
        self.decodeur = nn.Sequential(
            nn.Linear(8 + 10, 64 * 8 * 8), nn.ReLU(),
            nn.Unflatten(1, (64, 8, 8)),
            nn.ConvTranspose2d(64, 32, 4, 2, 1), nn.ReLU(),
            nn.ConvTranspose2d(32, 3, 4, 2, 1), nn.Sigmoid()
        )

    # Le one-hot a 10 valeurs ; une seule vaut 1, celle de la classe demandée.
    def encoder(self, x, y):
        h = self.encodeur(x)
        condition = F.one_hot(y, num_classes=10).float()
        h = torch.cat([h, condition], dim=1)
        h = F.relu(self.cachee(h))
        return self.moyenne(h), self.log_variance(h)

    # La classe accompagne aussi le code z lors de la génération.
    def decoder(self, z, y):
        condition = F.one_hot(y, num_classes=10).float()
        entree = torch.cat([z, condition], dim=1)
        return self.decodeur(entree)

    def forward(self, x, y):
        mu, logvar = self.encoder(x, y)
        sigma = torch.exp(0.5 * logvar)
        z = mu + sigma * torch.randn_like(mu)
        return self.decoder(z, y), mu, logvar

In [ ]:
# Somme des erreurs par image, puis moyenne sur le lot, pour les deux termes.
def vae_loss(reconstruction, images, mu, logvar):
    bce = F.binary_cross_entropy(reconstruction, images, reduction="sum") / len(images)
    kl = -0.5 * torch.sum(1 + logvar - mu ** 2 - logvar.exp()) / len(images)
    total = bce + kl
    return total, bce, kl

In [ ]:
# Optimiser la reconstruction ET la régularisation du code latent.
modele = CVAE().to(device)
optimiseur = torch.optim.Adam(modele.parameters(), lr=0.001)
historique = []
for epoque in range(20):
    modele.train()
    total_bce = 0.0
    total_kl = 0.0
    for images, etiquettes in chargeur:
        images = images.to(device)
        etiquettes = etiquettes.to(device)
        optimiseur.zero_grad()
        reconstruction, mu, logvar = modele(images, etiquettes)
        erreur, bce, kl = vae_loss(reconstruction, images, mu, logvar)
        # Calculer les gradients, puis modifier les poids.
        erreur.backward()
        optimiseur.step()
        total_bce += bce.item() * len(images)
        total_kl += kl.item() * len(images)
    ligne = {"BCE": total_bce / len(x_train), "KL": total_kl / len(x_train)}
    historique.append(ligne)
    print("CVAE", "époque", epoque + 1, ligne, flush=True)
modele.eval()
torch.save(modele.state_dict(), racine / "models" / "CVAE.pt")

In [ ]:
# Afficher les deux composantes de la perte conditionnelle.
plt.figure(figsize=(6, 3))
plt.plot([ligne["BCE"] for ligne in historique], label="BCE")
plt.plot([ligne["KL"] for ligne in historique], label="KL")
plt.xlabel("Époque (indice à partir de 0)")
plt.legend()
plt.tight_layout()
chemin = racine / "figures" / "C_pertes.png"
plt.savefig(chemin, dpi=140)
plt.close()
display(Image(filename=str(chemin)))

## Même z, différentes classes
On garde les mêmes huit codes aléatoires pour toutes les classes. Une ligne par
classe permet de voir ce qui change réellement quand on modifie y.

In [ ]:
# Générer huit images par classe, avec les mêmes codes dans chaque colonne.
codes_fixes = torch.randn(8, 8, device=device)
toutes_images = []
with torch.no_grad():
    for numero in range(10):
        y = torch.full((8,), numero, dtype=torch.long, device=device)
        images = modele.decoder(codes_fixes, y)
        toutes_images.append(images.cpu())
afficher(torch.cat(toutes_images),
         "CVAE : avion, voiture, oiseau, chat, cerf, chien, grenouille, cheval, bateau, camion",
         "C_classes.png")
print("Ordre des lignes :", classes)

## Réponses aux trois questions C
1. y permet de demander une catégorie ; le VAE standard ne dispose pas de cette entrée.
2. Si le modèle ignore y ou si la concaténation est incorrecte, changer la condition
peut ne pas modifier la classe apparente de l'image.
3. Une belle image ne prouve pas qu'elle respecte y. Il faut considérer conformité,
diversité et qualité. Ici, la conformité est inspectée qualitativement, sans annoncer
une précision numérique non mesurée. Les formes CIFAR peuvent rester ambiguës.

In [ ]:
# Sauvegarder le protocole et les pertes effectivement calculées.
resultats = {"historique": historique, "latent": 8, "epochs": 20,
             "classes": classes, "secondes": time.time() - debut}
(racine / "resultats_C.json").write_text(json.dumps(resultats, indent=2), encoding="utf-8")
print("Bloc C terminé.")